# Fine-tuning Vietnamese Bi-Encoder V2: Test Set + Hard Negative Mining

Notebook này là bản nâng cấp của `bi-encoder-2.ipynb` cho đồ án CS431.

Điểm mới:

- Tách dữ liệu thành `train / val / test` theo `(source, chapter)` để giảm leakage.
- Dùng `val` để chọn checkpoint.
- Chỉ dùng `test` để báo cáo kết quả cuối.
- Train vòng 1 bằng `MultipleNegativesRankingLoss`.
- Dùng model vòng 1 để mine hard negatives trên train set.
- Train vòng 2 với hard negatives đã mine.
- So sánh `base`, `v1`, `v2` trên test set độc lập.

## Kaggle setup

- Bật GPU T4.
- Bật Internet.
- Add dataset `dangvy1507/training-data` có các thư mục `training_data_*/qa_audit.jsonl`.


In [1]:
# %pip install -U "sentence-transformers>=3.0.0" datasets torch accelerate --quiet

## 1. Import và cấu hình

In [2]:
import os
import json
import math
import random
import time
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import torch
from datasets import Dataset
from sentence_transformers import SentenceTransformer
from sentence_transformers.losses import MultipleNegativesRankingLoss
from sentence_transformers.evaluation import InformationRetrievalEvaluator
from sentence_transformers import SentenceTransformerTrainer, SentenceTransformerTrainingArguments

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

MODEL_NAME = "bkai-foundation-models/vietnamese-bi-encoder"

# Kaggle dataset path from your previous notebook
KAGGLE_TRAINING_BASE = Path("/kaggle/input/datasets/dangvy1507/training-data")

if Path("/kaggle").is_dir():
    REPO_ROOT = Path("/kaggle/working")
    TRAINING_BASE = KAGGLE_TRAINING_BASE
    print("Environment: Kaggle")
else:
    REPO_ROOT = Path.cwd()
    TRAINING_BASE = REPO_ROOT / "data" / "training_data"
    print("Environment: local")

if not TRAINING_BASE.is_dir():
    # fallback for local repo layout: data/training_data/training_data_* or data/training_data_*
    alt = REPO_ROOT / "data"
    if alt.is_dir():
        TRAINING_BASE = alt

AUDIT_FILES = sorted(TRAINING_BASE.glob("**/qa_audit.jsonl"))
if not AUDIT_FILES:
    raise FileNotFoundError(f"Không tìm thấy qa_audit.jsonl dưới {TRAINING_BASE}")

OUTPUT_BASE = REPO_ROOT / "models" / "bi_encoder_hnm_v2"
CKPT_V1 = OUTPUT_BASE / "checkpoints_v1"
CKPT_V2 = OUTPUT_BASE / "checkpoints_v2"
MODEL_V1_DIR = OUTPUT_BASE / "vietnamese-bi-encoder-v1"
MODEL_V2_DIR = OUTPUT_BASE / "vietnamese-bi-encoder-v2-hnm"
ARTIFACT_DIR = OUTPUT_BASE / "artifacts"
for d in [CKPT_V1, CKPT_V2, MODEL_V1_DIR, MODEL_V2_DIR, ARTIFACT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Split ratios by chapter key
TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
TEST_RATIO = 0.15

# Stage 1 training
EPOCHS_V1 = 3
LR_V1 = 1e-5

# Stage 2 training after hard negative mining
EPOCHS_V2 = 2
LR_V2 = 5e-6

BATCH_SIZE = 32
WARMUP_RATIO = 0.10
WEIGHT_DECAY = 0.01
EVAL_STEPS = 100
SAVE_STEPS = 100
LOGGING_STEPS = 20

# Hard negative mining
MINE_TOP_K = 30
MINE_BATCH_SIZE = 64

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

print("\nAudit files:")
for p in AUDIT_FILES:
    print(" -", p)
print(f"\nOutput: {OUTPUT_BASE}")

Environment: Kaggle
Device: cuda
GPU: Tesla T4
VRAM: 15.6 GB

Audit files:
 - /kaggle/input/datasets/dangvy1507/training-data/training_data_lsd/qa_audit.jsonl
 - /kaggle/input/datasets/dangvy1507/training-data/training_data_pldc/qa_audit.jsonl
 - /kaggle/input/datasets/dangvy1507/training-data/training_data_th/qa_audit.jsonl

Output: /kaggle/working/models/bi_encoder_hnm_v2


## 2. Load data và split train/val/test theo chapter

In [3]:
def load_jsonl(path: Path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


def load_all_audits(paths):
    rows = []
    for path in paths:
        source = path.parent.name
        for row in load_jsonl(path):
            row["_source"] = source
            rows.append(row)
    return rows


def chapter_key(row):
    return (row.get("_source", ""), row.get("chapter", "Unknown"))


def split_by_chapter_3way(rows, train_ratio=0.70, val_ratio=0.15, seed=42):
    by_key = defaultdict(list)
    for r in rows:
        by_key[chapter_key(r)].append(r)

    keys = list(by_key.keys())
    rng = random.Random(seed)
    rng.shuffle(keys)

    n_total = len(rows)
    val_target = int(n_total * val_ratio)
    test_target = int(n_total * TEST_RATIO)

    train_rows, val_rows, test_rows = [], [], []
    for key in keys:
        bucket = by_key[key]
        if len(val_rows) < val_target:
            val_rows.extend(bucket)
        elif len(test_rows) < test_target:
            test_rows.extend(bucket)
        else:
            train_rows.extend(bucket)
    return train_rows, val_rows, test_rows


all_pairs = load_all_audits(AUDIT_FILES)
train_pairs, val_pairs, test_pairs = split_by_chapter_3way(
    all_pairs, TRAIN_RATIO, VAL_RATIO, RANDOM_SEED
)

print(f"Total pairs: {len(all_pairs)}")
print(f"Train      : {len(train_pairs)} ({len(train_pairs)/len(all_pairs)*100:.1f}%)")
print(f"Val        : {len(val_pairs)} ({len(val_pairs)/len(all_pairs)*100:.1f}%)")
print(f"Test       : {len(test_pairs)} ({len(test_pairs)/len(all_pairs)*100:.1f}%)")

split_keys = {
    "train": set(chapter_key(r) for r in train_pairs),
    "val": set(chapter_key(r) for r in val_pairs),
    "test": set(chapter_key(r) for r in test_pairs),
}
print("\nChapter-key overlap checks:")
print(" train & val :", len(split_keys["train"] & split_keys["val"]))
print(" train & test:", len(split_keys["train"] & split_keys["test"]))
print(" val & test  :", len(split_keys["val"] & split_keys["test"]))

print("\nIntent distribution:")
for name, rows in [("train", train_pairs), ("val", val_pairs), ("test", test_pairs)]:
    print(name, Counter(r.get("intent_type", "unknown") for r in rows))

print("\nSample:")
s = train_pairs[0]
print("Query   :", s.get("query", "")[:200])
print("Positive:", s.get("positive", "")[:200])


Total pairs: 4014
Train      : 2787 (69.4%)
Val        : 616 (15.3%)
Test       : 611 (15.2%)

Chapter-key overlap checks:
 train & val : 0
 train & test: 0
 val & test  : 0

Intent distribution:
train Counter({'relational': 976, 'applicative': 954, 'factual': 857})
val Counter({'relational': 218, 'applicative': 211, 'factual': 187})
test Counter({'relational': 212, 'applicative': 208, 'factual': 191})

Sample:
Query   : Tại sao các học thuyết duy tâm thường được tôn giáo sử dụng làm cơ sở lý luận?
Positive: + Chủ nghĩa duy tâm khách quan cũng thừa nhận tính thứ nhất của ý thức nhưng theo họ đấy là là thứ tinh thần khách quan có trước và tồn tại độc lập với con người. Thực thể tinh thần khách quan này thư


## 3. Dataset helpers và evaluator

In [4]:
def clean_text(text):
    return " ".join(str(text or "").split())


def pairs_to_dataset(rows, use_negative=False):
    records = []
    for p in rows:
        q = clean_text(p.get("query"))
        pos = clean_text(p.get("positive"))
        if not q or not pos:
            continue
        rec = {"anchor": q, "positive": pos}
        if use_negative:
            neg = clean_text(p.get("mined_negative") or p.get("negative"))
            if neg:
                rec["negative"] = neg
        records.append(rec)
    return Dataset.from_list(records)


def make_ir_eval(rows, all_rows, name):
    queries = {}
    corpus = {}
    relevant_docs = {}

    # one positive pid per query
    for i, p in enumerate(rows):
        qid = f"{name}_q_{i}"
        pid = f"{name}_p_{i}"
        queries[qid] = clean_text(p["query"])
        corpus[pid] = clean_text(p["positive"])
        relevant_docs[qid] = {pid}

    # add all positives as distractor corpus
    existing = set(corpus.values())
    for j, p in enumerate(all_rows):
        text = clean_text(p.get("positive"))
        if text and text not in existing:
            corpus[f"corpus_{j}"] = text
            existing.add(text)

    evaluator = InformationRetrievalEvaluator(
        queries=queries,
        corpus=corpus,
        relevant_docs=relevant_docs,
        name=name,
        precision_recall_at_k=[1, 5, 10],
        mrr_at_k=[10],
        show_progress_bar=False,
    )
    return evaluator, queries, corpus, relevant_docs


def evaluator_main_score(result):
    if isinstance(result, dict):
        for k, v in result.items():
            if "mrr@10" in k.lower():
                return float(v)
        return float(next(iter(result.values())))
    return float(result)


def metric_key_for_best(result):
    if isinstance(result, dict):
        for k in result:
            if "mrr@10" in k.lower():
                return f"eval_{k}"
    return "eval_val_cosine_mrr@10"


def flatten_metrics(prefix, result):
    if not isinstance(result, dict):
        return {f"{prefix}_mrr@10": float(result)}
    out = {}
    for k, v in result.items():
        out[f"{prefix}_{k}"] = float(v)
    return out


train_dataset_v1 = pairs_to_dataset(train_pairs, use_negative=False)
val_dataset = pairs_to_dataset(val_pairs, use_negative=False)
test_dataset = pairs_to_dataset(test_pairs, use_negative=False)

val_evaluator, val_queries, val_corpus, val_relevant = make_ir_eval(val_pairs, all_pairs, "val")
test_evaluator, test_queries, test_corpus, test_relevant = make_ir_eval(test_pairs, all_pairs, "test")

print(train_dataset_v1)
print(val_dataset)
print(test_dataset)
print(f"Val evaluator : queries={len(val_queries)}, corpus={len(val_corpus)}")
print(f"Test evaluator: queries={len(test_queries)}, corpus={len(test_corpus)}")

Dataset({
    features: ['anchor', 'positive'],
    num_rows: 2787
})
Dataset({
    features: ['anchor', 'positive'],
    num_rows: 616
})
Dataset({
    features: ['anchor', 'positive'],
    num_rows: 611
})
Val evaluator : queries=616, corpus=1836
Test evaluator: queries=611, corpus=1840


## 4. Baseline evaluation

In [5]:
base_model = SentenceTransformer(MODEL_NAME)
print(f"Loaded base model: {MODEL_NAME}")
print(f"Embedding dim: {base_model.get_sentence_embedding_dimension()}")
print(f"Device: {base_model.device}")

print("\nEvaluating base on val...")
base_val_result = val_evaluator(base_model, output_path=None)
base_val_mrr = evaluator_main_score(base_val_result)
METRIC_FOR_BEST = metric_key_for_best(base_val_result)

print("Evaluating base on test...")
base_test_result = test_evaluator(base_model, output_path=None)
base_test_mrr = evaluator_main_score(base_test_result)

print(f"Base val  MRR@10: {base_val_mrr:.4f}")
print(f"Base test MRR@10: {base_test_mrr:.4f}")
print(f"metric_for_best_model: {METRIC_FOR_BEST}")

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/777 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/167 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

bpe.codes: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/270 [00:00<?, ?B/s]

Loaded base model: bkai-foundation-models/vietnamese-bi-encoder
Embedding dim: 768
Device: cuda:0

Evaluating base on val...
Evaluating base on test...
Base val  MRR@10: 0.4234
Base test MRR@10: 0.4178
metric_for_best_model: eval_val_cosine_mrr@10


## 5. Train vòng 1

In [6]:
def train_sentence_transformer(model, train_dataset, eval_dataset, evaluator, output_dir, epochs, lr, run_name):
    loss = MultipleNegativesRankingLoss(model)
    steps_per_epoch = max(1, len(train_dataset) // BATCH_SIZE)
    total_steps = steps_per_epoch * epochs
    warmup_steps = int(total_steps * WARMUP_RATIO)

    print(f"[{run_name}] train samples: {len(train_dataset)}")
    print(f"[{run_name}] steps/epoch : {steps_per_epoch}")
    print(f"[{run_name}] total steps : {total_steps}")
    print(f"[{run_name}] warmup     : {warmup_steps}")

    args = SentenceTransformerTrainingArguments(
        output_dir=str(output_dir),
        num_train_epochs=epochs,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        learning_rate=lr,
        weight_decay=WEIGHT_DECAY,
        warmup_steps=warmup_steps,
        eval_strategy="steps",
        eval_steps=EVAL_STEPS,
        save_strategy="steps",
        save_steps=SAVE_STEPS,
        save_total_limit=2,
        load_best_model_at_end=True,
        metric_for_best_model=METRIC_FOR_BEST,
        greater_is_better=True,
        logging_strategy="steps",
        logging_steps=LOGGING_STEPS,
        logging_first_step=True,
        logging_dir=str(output_dir / "logs"),
        report_to="none",
        fp16=torch.cuda.is_available(),
        dataloader_num_workers=0,
        seed=RANDOM_SEED,
        disable_tqdm=False,
    )

    trainer = SentenceTransformerTrainer(
        model=model,
        args=args,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        loss=loss,
        evaluator=evaluator,
    )
    trainer.train()
    return model, trainer


model_v1, trainer_v1 = train_sentence_transformer(
    base_model,
    train_dataset_v1,
    val_dataset,
    val_evaluator,
    CKPT_V1,
    EPOCHS_V1,
    LR_V1,
    "v1",
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.
Currently using DataParallel (DP) for multi-gpu training, while DistributedDataParallel (DDP) is recommended for faster training. See https://sbert.net/docs/sentence_transformer/training/distributed.html for more information.


[v1] train samples: 2787
[v1] steps/epoch : 87
[v1] total steps : 261
[v1] warmup     : 26


Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

Step,Training Loss,Validation Loss,Val Cosine Accuracy@1,Val Cosine Accuracy@3,Val Cosine Accuracy@5,Val Cosine Accuracy@10,Val Cosine Precision@1,Val Cosine Precision@5,Val Cosine Precision@10,Val Cosine Recall@1,Val Cosine Recall@5,Val Cosine Recall@10,Val Cosine Ndcg@10,Val Cosine Mrr@10,Val Cosine Map@100
100,0.274646,0.601740,0.227273,0.717532,0.806818,0.899351,0.227273,0.161364,0.089935,0.227273,0.806818,0.899351,0.572519,0.465799,0.471081


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

## 6. Đánh giá vòng 1 trên val/test

In [7]:
print("Evaluating v1 on val...")
v1_val_result = val_evaluator(model_v1, output_path=None)
v1_val_mrr = evaluator_main_score(v1_val_result)

print("Evaluating v1 on test...")
v1_test_result = test_evaluator(model_v1, output_path=None)
v1_test_mrr = evaluator_main_score(v1_test_result)

print(f"V1 val  MRR@10: {v1_val_mrr:.4f}")
print(f"V1 test MRR@10: {v1_test_mrr:.4f}")

Evaluating v1 on val...
Evaluating v1 on test...
V1 val  MRR@10: 0.4658
V1 test MRR@10: 0.4795


## 7. Hard Negative Mining

Với mỗi query trong train set, dùng model vòng 1 retrieve top-k trong corpus train. Chọn đoạn sai có score cao nhất làm `mined_negative`. Đây là negative khó hơn random/BM25 negative ban đầu.

In [8]:
def mine_hard_negatives(model, train_rows, top_k=30, batch_size=64):
    # unique corpus from train positives
    corpus_texts = []
    text_to_idx = {}
    for p in train_rows:
        text = clean_text(p.get("positive"))
        if text and text not in text_to_idx:
            text_to_idx[text] = len(corpus_texts)
            corpus_texts.append(text)

    queries = [clean_text(p.get("query")) for p in train_rows]
    positive_texts = [clean_text(p.get("positive")) for p in train_rows]

    print(f"Mining corpus size: {len(corpus_texts)}")
    print(f"Mining queries    : {len(queries)}")

    t0 = time.time()
    corpus_emb = model.encode(
        corpus_texts,
        normalize_embeddings=True,
        show_progress_bar=True,
        batch_size=batch_size,
    )
    query_emb = model.encode(
        queries,
        normalize_embeddings=True,
        show_progress_bar=True,
        batch_size=batch_size,
    )

    mined_rows = []
    same_positive_count = 0
    no_negative_count = 0

    for i, p in enumerate(train_rows):
        scores = query_emb[i] @ corpus_emb.T
        k = min(top_k, len(corpus_texts))
        top_idx = np.argpartition(-scores, range(k))[:k]
        top_idx = top_idx[np.argsort(-scores[top_idx])]

        pos_text = positive_texts[i]
        neg_text = None
        neg_score = None
        for idx in top_idx:
            candidate = corpus_texts[int(idx)]
            if candidate == pos_text:
                same_positive_count += 1
                continue
            neg_text = candidate
            neg_score = float(scores[int(idx)])
            break

        if neg_text is None:
            existing_negs = p.get("negatives") or []
            neg_text = clean_text(existing_negs[0]) if existing_negs else None
            no_negative_count += 1

        new_p = dict(p)
        if neg_text:
            new_p["mined_negative"] = neg_text
            new_p["mined_negative_score"] = neg_score
        mined_rows.append(new_p)

    print(f"Mining done in {time.time() - t0:.1f}s")
    print(f"Fallback no-negative count: {no_negative_count}")
    return mined_rows


train_pairs_mined = mine_hard_negatives(model_v1, train_pairs, MINE_TOP_K, MINE_BATCH_SIZE)

# Save mined pairs for reproducibility
mined_path = ARTIFACT_DIR / "train_pairs_with_mined_negatives.jsonl"
with open(mined_path, "w", encoding="utf-8") as f:
    for row in train_pairs_mined:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")
print(f"Saved mined negatives: {mined_path}")

train_dataset_v2 = pairs_to_dataset(train_pairs_mined, use_negative=True)
print(train_dataset_v2)
print(train_dataset_v2[0].keys())
print("Sample mined negative:", train_dataset_v2[0].get("negative", "")[:200])

Mining corpus size: 1008
Mining queries    : 2787


Batches:   0%|          | 0/16 [00:00<?, ?it/s]

Batches:   0%|          | 0/44 [00:00<?, ?it/s]

Mining done in 21.4s
Fallback no-negative count: 0
Saved mined negatives: /kaggle/working/models/bi_encoder_hnm_v2/artifacts/train_pairs_with_mined_negatives.jsonl
Dataset({
    features: ['anchor', 'positive', 'negative'],
    num_rows: 2787
})
dict_keys(['anchor', 'positive', 'negative'])
Sample mined negative: Còn chủ nghĩa duy tâm triết học lại là sản phẩm của tư duy lý tính dựa trên cơ sở tri thức và lý trí. Về phương diện nhận thức luận, sai lầm của chủ nghĩa duy tâm bắt nguồn từ cách xem xét phiến diện,


## 8. Train vòng 2 với mined hard negatives

In [9]:
model_v2, trainer_v2 = train_sentence_transformer(
    model_v1,
    train_dataset_v2,
    val_dataset,
    val_evaluator,
    CKPT_V2,
    EPOCHS_V2,
    LR_V2,
    "v2_hnm",
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.
Currently using DataParallel (DP) for multi-gpu training, while DistributedDataParallel (DDP) is recommended for faster training. See https://sbert.net/docs/sentence_transformer/training/distributed.html for more information.


[v2_hnm] train samples: 2787
[v2_hnm] steps/epoch : 87
[v2_hnm] total steps : 174
[v2_hnm] warmup     : 17


Step,Training Loss,Validation Loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

## 9. Final evaluation trên test set độc lập

In [10]:
print("Evaluating v2 on val...")
v2_val_result = val_evaluator(model_v2, output_path=None)
v2_val_mrr = evaluator_main_score(v2_val_result)

print("Evaluating v2 on test...")
v2_test_result = test_evaluator(model_v2, output_path=None)
v2_test_mrr = evaluator_main_score(v2_test_result)

summary_rows = [
    {"model": "base", "val_mrr@10": base_val_mrr, "test_mrr@10": base_test_mrr},
    {"model": "v1_mnrl", "val_mrr@10": v1_val_mrr, "test_mrr@10": v1_test_mrr},
    {"model": "v2_hard_negative", "val_mrr@10": v2_val_mrr, "test_mrr@10": v2_test_mrr},
]

print("\nMRR@10 summary")
print("=" * 72)
print(f"{'model':<22} {'val_mrr@10':>12} {'test_mrr@10':>13} {'test_delta_base':>16}")
for row in summary_rows:
    delta = row["test_mrr@10"] - base_test_mrr
    print(f"{row['model']:<22} {row['val_mrr@10']:>12.4f} {row['test_mrr@10']:>13.4f} {delta:>+16.4f}")
    
full_metrics = {
    "base_val": flatten_metrics("base_val", base_val_result),
    "base_test": flatten_metrics("base_test", base_test_result),
    "v1_val": flatten_metrics("v1_val", v1_val_result),
    "v1_test": flatten_metrics("v1_test", v1_test_result),
    "v2_val": flatten_metrics("v2_val", v2_val_result),
    "v2_test": flatten_metrics("v2_test", v2_test_result),
}

with open(ARTIFACT_DIR / "bi_encoder_hnm_eval_summary.json", "w", encoding="utf-8") as f:
    json.dump({"summary": summary_rows, "metrics": full_metrics}, f, ensure_ascii=False, indent=2)

csv_path = ARTIFACT_DIR / "bi_encoder_hnm_eval_summary.csv"
with open(csv_path, "w", encoding="utf-8") as f:
    f.write("model,val_mrr@10,test_mrr@10,test_delta_base\n")
    for row in summary_rows:
        delta = row["test_mrr@10"] - base_test_mrr
        f.write(f"{row['model']},{row['val_mrr@10']:.6f},{row['test_mrr@10']:.6f},{delta:.6f}\n")

print(f"\nSaved metrics: {ARTIFACT_DIR / 'bi_encoder_hnm_eval_summary.json'}")
print(f"Saved CSV    : {csv_path}")

Evaluating v2 on val...
Evaluating v2 on test...

MRR@10 summary
model                    val_mrr@10   test_mrr@10  test_delta_base
base                         0.4234        0.4178          +0.0000
v1_mnrl                      0.4658        0.4795          +0.0617
v2_hard_negative             0.4864        0.4982          +0.0805

Saved metrics: /kaggle/working/models/bi_encoder_hnm_v2/artifacts/bi_encoder_hnm_eval_summary.json
Saved CSV    : /kaggle/working/models/bi_encoder_hnm_v2/artifacts/bi_encoder_hnm_eval_summary.csv


## 10. Save models và metadata

In [11]:
model_v1.save(str(MODEL_V1_DIR))
model_v2.save(str(MODEL_V2_DIR))

metadata = {
    "base_model": MODEL_NAME,
    "audit_files": [str(p) for p in AUDIT_FILES],
    "split": {
        "train_pairs": len(train_pairs),
        "val_pairs": len(val_pairs),
        "test_pairs": len(test_pairs),
        "split_unit": "(source, chapter)",
    },
    "stage_1": {
        "epochs": EPOCHS_V1,
        "lr": LR_V1,
        "loss": "MultipleNegativesRankingLoss",
    },
    "stage_2": {
        "epochs": EPOCHS_V2,
        "lr": LR_V2,
        "loss": "MultipleNegativesRankingLoss with mined hard negatives",
        "mine_top_k": MINE_TOP_K,
    },
    "mrr@10": {
        "base_val": base_val_mrr,
        "base_test": base_test_mrr,
        "v1_val": v1_val_mrr,
        "v1_test": v1_test_mrr,
        "v2_val": v2_val_mrr,
        "v2_test": v2_test_mrr,
    },
}

for out_dir in [MODEL_V1_DIR, MODEL_V2_DIR]:
    with open(out_dir / "training_metadata.json", "w", encoding="utf-8") as f:
        json.dump(metadata, f, ensure_ascii=False, indent=2)

print(f"Saved V1 model: {MODEL_V1_DIR}")
print(f"Saved V2 model: {MODEL_V2_DIR}")
print("\nUse V2 model in RAG:")
print(f"from sentence_transformers import SentenceTransformer")
print(f"model = SentenceTransformer('{MODEL_V2_DIR.resolve()}')")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved V1 model: /kaggle/working/models/bi_encoder_hnm_v2/vietnamese-bi-encoder-v1
Saved V2 model: /kaggle/working/models/bi_encoder_hnm_v2/vietnamese-bi-encoder-v2-hnm

Use V2 model in RAG:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer('/kaggle/working/models/bi_encoder_hnm_v2/vietnamese-bi-encoder-v2-hnm')
